In [0]:
import logging
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lit, when, row_number, to_timestamp, year, month, current_timestamp
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("metyis_pipeline")

SOURCE_PATH               = "/Volumes/workspace/default/sales_data/raw/Sales_Data_2019/"
CLEANSED_PATH             = "/Volumes/workspace/default/sales_data/cleansed/"
CLEANSED_PARTITIONED_PATH = CLEANSED_PATH + "sales_partitioned/"
CLEANSED_COMBINED_PATH    = CLEANSED_PATH + "sales_combined/"
QUARANTINE_PATH           = "/Volumes/workspace/default/sales_data/quarantine/"

In [0]:
files = dbutils.fs.ls(SOURCE_PATH)
for f in files:
    print(f.name)

Sales_April_2019.csv
Sales_August_2019.csv
Sales_December_2019.csv
Sales_February_2019.csv
Sales_January_2019.csv
Sales_July_2019.csv
Sales_June_2019.csv
Sales_March_2019.csv
Sales_May_2019.csv
Sales_November_2019.csv
Sales_October_2019.csv
Sales_September_2019.csv


In [0]:
# Check the schema of the first file

df_jan = spark.read.csv(    
                        SOURCE_PATH + "Sales_January_2019.csv",
                        header=True
)

df_jan.show(10)
print("Rows",df_jan.count())
df_jan.printSchema()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  141234|              iPhone|               1|       700|01/22/19 21:25|944 Walnut St, Bo...|
|  141235|Lightning Chargin...|               1|     14.95|01/28/19 14:15|185 Maple St, Por...|
|  141236|    Wired Headphones|               2|     11.99|01/17/19 13:33|538 Adams St, San...|
|  141237|    27in FHD Monitor|               1|    149.99|01/05/19 20:33|738 10th St, Los ...|
|  141238|    Wired Headphones|               1|     11.99|01/25/19 11:59|387 10th St, Aust...|
|  141239|AAA Batteries (4-...|               1|      2.99|01/29/19 20:22|775 Willow St, Sa...|
|  141240|27in 4K Gaming Mo...|               1|    389.99|01/26/19 12:16|979 Park St, Los ...|
|  141241|USB-C Charging Cable|         

In [0]:
try:
    logger.info("Stage 1: Reading raw source files")
    df_raw = spark.read.csv(SOURCE_PATH, header=True)
    logger.info(f"Read {df_raw.count()} total rows across all files")
except Exception as e:
    logger.error(f"Stage 1 failed: {e}")
    raise

print("Total rows across all 12 files:", df_raw.count())

INFO:metyis_pipeline:Stage 1: Reading raw source files
INFO:metyis_pipeline:Read 186850 total rows across all files


Total rows across all 12 files: 186850


In [0]:
df_raw.describe().show()

+-------+------------------+------------+------------------+------------------+--------------+--------------------+
|summary|          Order ID|     Product|  Quantity Ordered|        Price Each|    Order Date|    Purchase Address|
+-------+------------------+------------+------------------+------------------+--------------+--------------------+
|  count|            186305|      186305|            186305|            186305|        186305|              186305|
|   mean| 230417.5693788653|        NULL|1.1243828986286637| 184.3997347674056|          NULL|                NULL|
| stddev|51512.737109996044|        NULL|0.4427926240286722|332.73132988434384|          NULL|                NULL|
|    min|            141234|20in Monitor|                 1|            109.99|01/01/19 03:07|1 11th St, Atlant...|
|    max|          Order ID|      iPhone|  Quantity Ordered|        Price Each|    Order Date|    Purchase Address|
+-------+------------------+------------+------------------+------------

In [0]:
from pyspark.sql.functions import col 

try:
    logger.info("Stage 2: Removing junk rows (headers/blanks)")
    header_rows = df_raw.filter(col("Order ID") == "Order ID")
    blank_rows = df_raw.filter(col("Order ID").isNull())
    df_clean = df_raw.filter(col("Order ID") != "Order ID").filter(col("Order ID").isNotNull())

    removed = df_raw.count() - df_clean.count()
    expected = header_rows.count() + blank_rows.count()
    if removed != expected:
        raise ValueError(f"Reconciliation mismatch: removed {removed}, expected {expected}")
    
    
    logger.info(f"Removed {removed} junk rows, reconciliation check passed")
    print("Embedded header rows:", header_rows.count())
    print("Blank rows:", blank_rows.count())
except Exception as e:
    logger.error(f"Stage 2 failed: {e}")
    raise

INFO:metyis_pipeline:Stage 2: Removing junk rows (headers/blanks)
INFO:metyis_pipeline:Removed 900 junk rows, reconciliation check passed


Embedded header rows: 355
Blank rows: 545


In [0]:
print("Null Quantity Ordered:", df_clean.filter(col("Quantity Ordered").isNull()).count())
print("Null Price Each:", df_clean.filter(col("Price Each").isNull()).count())
print("Null Order Date:", df_clean.filter(col("Order Date").isNull()).count())
print("Null Purchase Address:", df_clean.filter(col("Purchase Address").isNull()).count())

Null Quantity Ordered: 0
Null Price Each: 0
Null Order Date: 0
Null Purchase Address: 0


In [0]:
total = df_clean.count()

# (a) fully identical rows: every of the 6 columns must be identical 

exact_dupes = total - df_clean.dropDuplicates().count()
print("Exact duplicate rows:", exact_dupes)

# (b) rows sharing justsame Order ID + Product 

key_dupes = total - df_clean.dropDuplicates(["Order ID", "Product"]).count()
print("Extra rows sharing Order ID + Product:", key_dupes)

print("Finding the same product appearing twice in one order", key_dupes - exact_dupes)

Exact duplicate rows: 264
Extra rows sharing Order ID + Product: 311
Finding the same product appearing twice in one order 47


In [0]:
try:
    logger.info("Stage 3: Removing exact duplicate rows")
    dedup_window = Window.partitionBy(df_clean.columns).orderBy("Order ID")

    df_deduped = (
        df_clean
        .withColumn("copy_number", row_number().over(dedup_window))
        .filter(col("copy_number") == 1)
        .drop("copy_number")
    )

    deduped_count = df_deduped.count()
    logger.info(f"Removed {df_clean.count() - deduped_count} exact duplicates, {deduped_count} rows remain")
except Exception as e:
    logger.error(f"Stage 3 failed: {e}")
    raise

INFO:metyis_pipeline:Stage 3: Removing exact duplicate rows
INFO:metyis_pipeline:Removed 264 exact duplicates, 185686 rows remain


In [0]:
from pyspark.sql.functions import countDistinct

conflict_keys = (
    df_deduped.groupBy("Order ID", "Product")
        .agg(
            countDistinct("Quantity Ordered", "Price Each", "Order Date", "Purchase Address")
              .alias("distinct_versions")
        )
        .filter(col("distinct_versions") > 1)
)

print("Conflict keys:", conflict_keys.count())
conflict_keys.show(20)

conflict_rows = df_deduped.join(
    conflict_keys.select("Order ID", "Product"),
    on=["Order ID", "Product"],
    how="inner"
).orderBy("Order ID", "Product")

conflict_rows.show(20)

Conflict keys: 47
+--------+--------------------+-----------------+
|Order ID|             Product|distinct_versions|
+--------+--------------------+-----------------+
|  186331|AAA Batteries (4-...|                2|
|  251220|    Wired Headphones|                2|
|  165732|AAA Batteries (4-...|                2|
|  160329|AAA Batteries (4-...|                2|
|  212216|AAA Batteries (4-...|                2|
|  240914|AA Batteries (4-p...|                2|
|  273666|AA Batteries (4-p...|                2|
|  309706|AA Batteries (4-p...|                2|
|  155031|Lightning Chargin...|                2|
|  163578|AA Batteries (4-p...|                2|
|  299605|AA Batteries (4-p...|                2|
|  306851|AA Batteries (4-p...|                2|
|  239659|AA Batteries (4-p...|                2|
|  173803|USB-C Charging Cable|                2|
|  284704|AA Batteries (4-p...|                2|
|  219631|USB-C Charging Cable|                2|
|  202271|    Wired Headphones| 

In [0]:
try:
    logger.info("Stage 4: Flagging conflicts, casting types, renaming columns")

    # Order lines that still appear more than once after dedup = conflicting values
    conflict_keys = (
        df_deduped
        .groupBy("Order ID", "Product").count()
        .filter(col("count") > 1)
        .select("Order ID", "Product")
        .withColumn("data_quality_flag", lit("quarantined"))
    )

    df_flagged = (
        df_deduped
        .join(conflict_keys, ["Order ID", "Product"], "left")
        .withColumn("data_quality_flag",
                    when(col("data_quality_flag").isNull(), "clean")
                    .otherwise(col("data_quality_flag")))
    )

    df_output = (
        df_flagged
        .withColumn("Quantity Ordered", col("Quantity Ordered").cast("int"))
        .withColumn("Price Each", col("Price Each").cast("decimal(10,2)"))
        .withColumn("Order Date", to_timestamp(col("Order Date"), "MM/dd/yy HH:mm"))
        .withColumn("year", year(col("Order Date")))
        .withColumn("month", month(col("Order Date")))
        .withColumnRenamed("Order ID", "order_id")
        .withColumnRenamed("Product", "product")
        .withColumnRenamed("Quantity Ordered", "quantity_ordered")
        .withColumnRenamed("Price Each", "price_each")
        .withColumnRenamed("Order Date", "order_date")
        .withColumnRenamed("Purchase Address", "purchase_address")
    )

    # A failed cast silently becomes null, so check every cast column
    for c in ["quantity_ordered", "price_each", "order_date"]:
        nulls = df_output.filter(col(c).isNull()).count()
        if nulls > 0:
            raise ValueError(f"{nulls} rows failed casting in column '{c}'")

    logger.info("All casts succeeded, no nulls introduced")
except Exception as e:
    logger.error(f"Stage 4 failed: {e}")
    raise

INFO:metyis_pipeline:Stage 4: Flagging conflicts, casting types, renaming columns
INFO:metyis_pipeline:All casts succeeded, no nulls introduced


In [0]:
try:
    logger.info("Stage 5: Splitting and writing outputs")

    df_clean_out = (
        df_output
        .filter(col("data_quality_flag") == "clean")
        .drop("data_quality_flag")
    )

    df_quarantine_out = (
        df_output
        .filter(col("data_quality_flag") == "quarantined")
        .drop("data_quality_flag")
        .withColumn("quarantine_reason", lit("conflicting_quantity_same_order_product"))
        .withColumn("quarantined_at", current_timestamp())
    )

    # Main output: partitioned, one file per partition folder
    (df_clean_out
        .repartition("year", "month")
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("year", "month")
        .format("delta")
        .save(CLEANSED_PARTITIONED_PATH))
    logger.info("Partitioned output written")

    # Literal 'single file' version of the same data
    (df_clean_out
        .coalesce(1)
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .save(CLEANSED_COMBINED_PATH))
    logger.info("Combined single-file output written")

    (df_quarantine_out.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .save(QUARANTINE_PATH))
    logger.info("Quarantine output written")
except Exception as e:
    logger.error(f"Stage 5 failed: {e}")
    raise

INFO:metyis_pipeline:Stage 5: Splitting and writing outputs
INFO:metyis_pipeline:Partitioned output written
INFO:metyis_pipeline:Combined single-file output written
INFO:metyis_pipeline:Quarantine output written


In [0]:
df_clean_out.show(10)

+--------+--------------------+----------------+----------+-------------------+--------------------+----+-----+
|order_id|             product|quantity_ordered|price_each|         order_date|    purchase_address|year|month|
+--------+--------------------+----------------+----------+-------------------+--------------------+----+-----+
|  141239|AAA Batteries (4-...|               1|      2.99|2019-01-29 20:22:00|775 Willow St, Sa...|2019|    1|
|  141245|  Macbook Pro Laptop|               1|   1700.00|2019-01-31 10:12:00|322 6th St, San F...|2019|    1|
|  141247|    27in FHD Monitor|               1|    149.99|2019-01-25 19:19:00|512 Wilson St, Sa...|2019|    1|
|  141257|Apple Airpods Hea...|               1|    150.00|2019-01-12 18:51:00|338 Highland St, ...|2019|    1|
|  141260|AAA Batteries (4-...|               1|      2.99|2019-01-01 22:00:00|293 Hill St, San ...|2019|    1|
|  141271|USB-C Charging Cable|               1|     11.95|2019-01-30 10:51:00|90 13th St, Bosto...|2019

In [0]:
try:
    logger.info("Stage 6: Validating written outputs")

    df_part = spark.read.format("delta").load(CLEANSED_PARTITIONED_PATH)
    df_comb = spark.read.format("delta").load(CLEANSED_COMBINED_PATH)
    df_quar = spark.read.format("delta").load(QUARANTINE_PATH)

    # Count once, reuse the numbers (each .count() is a full pass over the data)
    part_count = df_part.count()
    comb_count = df_comb.count()
    quar_count = df_quar.count()

    # 1) Nothing lost or double-counted: clean + quarantine = deduped total
    if part_count + quar_count != deduped_count:
        raise ValueError(f"Reconciliation failed: {part_count} + {quar_count} != {deduped_count}")

    # 2) Both cleansed versions hold the same data
    if part_count != comb_count:
        raise ValueError(f"Partitioned ({part_count}) and combined ({comb_count}) row counts differ")

    # 3) Combined version really is one file (current Delta version only)
    num_files = spark.sql(f"DESCRIBE DETAIL delta.`{CLEANSED_COMBINED_PATH}`").collect()[0]["numFiles"]
    if num_files != 1:
        raise ValueError(f"Combined output has {num_files} files, expected 1")

    # 4) Cleansed has exactly one row per order line
    dup_keys = (
        df_part
        .groupBy("order_id", "product").count()
        .filter(col("count") > 1)
        .count()
    )
    if dup_keys > 0:
        raise ValueError(f"{dup_keys} duplicate business keys found in cleansed output")

    logger.info(f"Reconciliation passed: {part_count} clean + {quar_count} quarantined = {deduped_count}")
    logger.info(f"Partitioned and combined outputs match ({part_count} rows); combined is 1 file")
    logger.info("No duplicate business keys in cleansed output")

except Exception as e:
    logger.error(f"Stage 6 failed: {e}")
    raise

INFO:metyis_pipeline:Stage 6: Validating written outputs
INFO:metyis_pipeline:Reconciliation passed: 185592 clean + 94 quarantined = 185686
INFO:metyis_pipeline:Partitioned and combined outputs match (185592 rows); combined is 1 file
INFO:metyis_pipeline:No duplicate business keys in cleansed output


In [0]:
# DEMO ONLY: "what if the business says keep the higher quantity?"
# Off by default
RUN_RESOLUTION_DEMO = False

if RUN_RESOLUTION_DEMO:
    from delta.tables import DeltaTable

    # 1) Pick one winner per conflict: the higher quantity
    resolve_window = Window.partitionBy("order_id", "product").orderBy(col("quantity_ordered").desc())

    df_resolved = (
        spark.read.format("delta").load(QUARANTINE_PATH)
        .withColumn("rn", row_number().over(resolve_window))
        .filter(col("rn") == 1)
        .drop("rn", "quarantine_reason", "quarantined_at")
    )

    # 2) Merge the winners into the partitioned cleansed output
    cleansed = DeltaTable.forPath(spark, CLEANSED_PARTITIONED_PATH)

    (cleansed.alias("t")
        .merge(
            df_resolved.alias("s"),
            "t.year = s.year AND t.month = s.month "
            "AND t.order_id = s.order_id AND t.product = s.product")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    # 3) Check
    df_after = spark.read.format("delta").load(CLEANSED_PARTITIONED_PATH)
    print("Cleansed rows:", df_after.count())   # expect 185,592 + 47 = 185,639
else:
    logger.info("Resolution demo skipped (RUN_RESOLUTION_DEMO = False)")

In [0]:
# Convenience export: CSV copies so can open the data without Spark (e.g. Excel).
# Fine at this size (~186K rows). At large volumes I'd skip this or export a sample.
EXPORT_PATH = "/Volumes/workspace/default/sales_data/exports/"
dbutils.fs.mkdirs(EXPORT_PATH)

(spark.read.format("delta").load(CLEANSED_COMBINED_PATH)
    .orderBy("order_date")
    .toPandas()
    .to_csv(EXPORT_PATH + "sales_cleansed.csv", index=False))

(spark.read.format("delta").load(QUARANTINE_PATH)
    .orderBy("order_id", "product")
    .toPandas()
    .to_csv(EXPORT_PATH + "sales_quarantine.csv", index=False))

logger.info("CSV exports written")